#### Save the best model

XGBoost was the best model for UCI at all horizons, and best or not significantly different from the best model for PJME
at 24h and 168h (notebook 06). It is also fast and small, so it is used in the prediction page of the dashboard.
Same tuned settings and training data as notebook 04 (PJME: last 4 years of train), seed 0. Saved to models/.

In [1]:
import os
import json

import numpy as np
import pandas as pd
import xgboost as xgb

In [2]:
HORIZONS = [1, 24, 168]
PJME_TRAIN_HOURS = 24 * 365 * 4
os.makedirs("../models", exist_ok=True)

with open("../results/baseline_params.json") as f:
    params = json.load(f)

Same features as notebook 04.

In [3]:
def make_features(df, h):
    X = pd.DataFrame(index=df.index)

    lags = [h + k for k in range(6)] + [m for m in [24, 48, 168, 336] if m >= h]
    for s in sorted(set(lags)):
        X[f"lag_{s}"] = df["load"].shift(s)

    X["roll_mean_24"] = df["load"].shift(h).rolling(24).mean()
    X["roll_mean_168"] = df["load"].shift(h).rolling(168).mean()

    for c in ["hour", "dayofweek", "month", "dayofyear", "weekend", "holiday"]:
        X[c] = df[c]

    X = X.dropna()
    y = df["load"].loc[X.index]
    return X, y

In [4]:
for name in ["pjme", "uci"]:
    train = pd.read_parquet(f"../data/processed/{name}_train.parquet")
    val = pd.read_parquet(f"../data/processed/{name}_val.parquet")
    test = pd.read_parquet(f"../data/processed/{name}_test.parquet")
    full = pd.concat([train, val, test])

    for h in HORIZONS:
        X, y = make_features(full, h)
        tr = X.index <= train.index.max()
        if name == "pjme":
            tr = tr & (X.index >= train.index[-PJME_TRAIN_HOURS])
        te = X.index >= test.index.min()

        model = xgb.XGBRegressor(**params[name]["xgb"], tree_method="hist", n_jobs=-1, random_state=0)
        model.fit(X[tr], y[tr])
        model.save_model(f"../models/xgb_{name}_h{h}.json")

        # check against the predictions saved in notebook 04
        old = pd.read_parquet(f"../results/predictions/{name}_xgb_h{h}.parquet")["seed_0"]
        new = model.predict(X[te])
        print(f"{name} h={h}: max difference to notebook 04 = {np.abs(new - old.values).max():.6f}")

pjme h=1: max difference to notebook 04 = 0.000000


pjme h=24: max difference to notebook 04 = 0.000000


pjme h=168: max difference to notebook 04 = 0.000000


uci h=1: max difference to notebook 04 = 0.000000


uci h=24: max difference to notebook 04 = 0.000000


uci h=168: max difference to notebook 04 = 0.000000


In [5]:
sorted(os.listdir("../models"))

['xgb_pjme_h1.json',
 'xgb_pjme_h168.json',
 'xgb_pjme_h24.json',
 'xgb_uci_h1.json',
 'xgb_uci_h168.json',
 'xgb_uci_h24.json']